# Parking-space occupancy: PyTorch baseline CNN

This notebook trains a binary classifier on individual parking-spot images. CNR manifest rows are `relative/path/to/image.jpg label`, where `0` is empty and `1` is occupied. Images are resolved under `PATCHES/`.


In [1]:
# If needed, uncomment and run once in this environment:
# %pip install torch pillow

from pathlib import Path
import random
import numpy as np
from PIL import Image, ImageEnhance
%pip install torch
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
IMAGE_SIZE = (150, 150)  # native patch resolution
BATCH_SIZE = 64
EPOCHS = 12
LEARNING_RATE = 1e-3
ROOT = Path.cwd()
PATCHES_DIR = ROOT / 'PATCHES'
LABELS_DIR = ROOT / 'LABELS'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
print('Dataset root:', ROOT.resolve())



[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
PyTorch: 2.15.0
Device: cpu
Dataset root: /Users/ashleynavos/Downloads/CNR-EXT-Patches-150x150/data-4620-project


In [2]:
def read_manifest(filename):
    """Read a CNR manifest and return absolute image paths and integer labels."""
    paths, labels = [], []
    with (LABELS_DIR / filename).open() as handle:
        for line_number, line in enumerate(handle, start=1):
            fields = line.strip().rsplit(maxsplit=1)
            if not fields:
                continue
            if len(fields) != 2:
                raise ValueError(f'{filename}:{line_number} is malformed: {line!r}')
            relative_path, label = fields
            image_path = PATCHES_DIR / relative_path
            if not image_path.is_file():
                raise FileNotFoundError(f'Missing image: {image_path}')
            paths.append(image_path)
            labels.append(int(label))
    return paths, labels

train_paths, train_labels = read_manifest('train.txt')
val_paths, val_labels = read_manifest('val.txt')
test_paths, test_labels = read_manifest('test.txt')

for name, labels in [('train', train_labels), ('validation', val_labels), ('test', test_labels)]:
    labels = np.asarray(labels)
    print(f'{name:10s}: {len(labels):6,d} images | empty (0): {(labels == 0).sum():6,d} | occupied (1): {(labels == 1).sum():6,d}')


FileNotFoundError: [Errno 2] No such file or directory: '/Users/ashleynavos/Downloads/CNR-EXT-Patches-150x150/data-4620-project/LABELS/train.txt'

In [ ]:
class ParkingDataset(Dataset):
    def __init__(self, paths, labels, training=False):
        self.paths = paths
        self.labels = labels
        self.training = training

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert('RGB').resize(IMAGE_SIZE)
            if self.training and random.random() < 0.5:
                image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            if self.training:
                image = ImageEnhance.Brightness(image).enhance(random.uniform(0.9, 1.1))
                image = ImageEnhance.Contrast(image).enhance(random.uniform(0.9, 1.1))
            array = np.asarray(image, dtype=np.float32) / 255.0

        # Convert HWC RGB image to CHW tensor expected by Conv2d.
        image_tensor = torch.from_numpy(array).permute(2, 0, 1)
        label_tensor = torch.tensor(self.labels[index], dtype=torch.float32)
        return image_tensor, label_tensor

train_dataset = ParkingDataset(train_paths, train_labels, training=True)
val_dataset = ParkingDataset(val_paths, val_labels)
test_dataset = ParkingDataset(test_paths, test_labels)

loader_kwargs = {'batch_size': BATCH_SIZE, 'num_workers': 0, 'pin_memory': DEVICE.type == 'cuda'}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)


In [ ]:
class ParkingCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm2d(128), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(128, 1))

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)  # raw logits

model = ParkingCNN().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


ParkingCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
    (10): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): AdaptiveAvgPool2d(output_size=1)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(

In [ ]:
def run_epoch(loader, training=False):
    model.train(training)
    total_loss = total_correct = total_examples = 0
    all_probabilities, all_labels = [], []

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            probabilities = torch.sigmoid(logits)
            total_loss += loss.item() * labels.size(0)
            total_correct += ((probabilities >= 0.5) == labels.bool()).sum().item()
            total_examples += labels.size(0)
            all_probabilities.append(probabilities.detach().cpu())
            all_labels.append(labels.detach().cpu())

    return (total_loss / total_examples, total_correct / total_examples,
            torch.cat(all_probabilities), torch.cat(all_labels))

best_val_loss, epochs_without_improvement = float('inf'), 0
PATIENCE = 3
for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy, _, _ = run_epoch(train_loader, training=True)
    val_loss, val_accuracy, _, _ = run_epoch(val_loader)
    print(f'Epoch {epoch:02d}/{EPOCHS} | train loss: {train_loss:.4f}, accuracy: {train_accuracy:.4f} | val loss: {val_loss:.4f}, accuracy: {val_accuracy:.4f}')

    if val_loss < best_val_loss:
        best_val_loss, epochs_without_improvement = val_loss, 0
        torch.save({'model_state_dict': model.state_dict(), 'image_size': IMAGE_SIZE}, 'parking_occupancy_baseline.pt')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('Early stopping.')
            break

checkpoint = torch.load('parking_occupancy_baseline.pt', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])


In [ ]:
test_loss, test_accuracy, probabilities, labels = run_epoch(test_loader)
predictions = (probabilities >= 0.5).to(torch.int)
labels_int = labels.to(torch.int)

true_positive = ((predictions == 1) & (labels_int == 1)).sum().item()
false_positive = ((predictions == 1) & (labels_int == 0)).sum().item()
false_negative = ((predictions == 0) & (labels_int == 1)).sum().item()
precision = true_positive / max(true_positive + false_positive, 1)
recall = true_positive / max(true_positive + false_negative, 1)

print(f'Test loss:      {test_loss:.4f}')
print(f'Test accuracy:  {test_accuracy:.4f}')
print(f'Test precision: {precision:.4f}')
print(f'Test recall:    {recall:.4f}')

# Probability >= 0.5 means the spot is classified as occupied.
for truth, probability in zip(labels_int[:8].tolist(), probabilities[:8].tolist()):
    print(f'true={truth} | predicted={int(probability >= 0.5)} | occupied probability={probability:.3f}')
